ベース：黒地図 CARTO Dark Matter

国境線：HDX OCHA 'Syrian Arab Republic - Subnational Administrative Boundaries'
https://data.humdata.org/dataset/cod-ab-syr
syr_admin0.geojson

地域線：HDX OCHA 'Syrian Arab Republic - Subnational Administrative Boundaries'
https://data.humdata.org/dataset/cod-ab-syr
syr_admin1.geojson

人口ラスタ：WorldPop 'Syrian Arab Republic 100m Population 2026'
https://hub.worldpop.org/geodata/summary?id=75632
syr_worldpop_2026.tif

分析手法：Zonal Statistics

対象地域：
・Aleppo
・Idleb
・Lattakia

その他：
・行政区ポリゴンと人口ラスタを重ね、各Governorate内の推定人口合計を集計
・集計結果をProportional Symbol Mapとして可視化

In [59]:
!pip install rasterstats

zsh:1: command not found: pip


In [60]:
# Core Libraries
import geopandas as gpd

import rasterio

import folium
from rasterstats import zonal_stats

In [61]:
# ETL Extract
# 行政区ポリゴンを読み込む
admin1 = gpd.read_file("syr_admin1.geojson")

# ETL Extract
# 人口ラスタのCRSと範囲を確認
with rasterio.open("tif_1_syria_worldpop_2026.tif") as src:
    print(src.crs)
    print(src.bounds)

EPSG:4326
BoundingBox(left=35.7166658038, bottom=32.310833540089995, right=42.37666577716, top=37.320000186719994)


In [62]:
# ETL Transform
# Zonal Statistics
# 対象Governorateごとに人口ラスタを集計
target_admins = admin1[
    admin1["adm1_name"].isin([
        "Aleppo",
        "Idleb",
        "Lattakia"
    ])
]

stats = zonal_stats(
    target_admins,
    "tif_1_syria_worldpop_2026.tif",
    stats=["sum"],
    geojson_out=True
)


zonal_stats() により、人口合計（sum）、Governorate情報、
geometry、polygon座標、metadataを含むデータを取得

In [63]:
# Check
# 集計結果を確認
for area in stats:
    print(area["properties"]["adm1_name"])
    print(area["properties"]["sum"])

Aleppo
4550203.0
Idleb
1746525.75
Lattakia
1519531.0


In [64]:
# ETL Transform
# Raster解析
# zonal statistics結果から人口合計を抽出

for area in stats:
    print(area["properties"]["adm1_name"])
    print(area["properties"]["sum"])

Aleppo
4550203.0
Idleb
1746525.75
Lattakia
1519531.0


In [65]:
# ETL Transform
# Folium描画用データへ整形
population_results = []

for area in stats:
    props = area["properties"]

    geom = target_admins[
        target_admins["adm1_name"] == props["adm1_name"]
    ].geometry.iloc[0]

    point = geom.representative_point()

    population_results.append({
        "name": props["adm1_name"],
        "population": props["sum"],
        "lat": point.y,
        "lon": point.x,
    })

print(population_results)

[{'name': 'Aleppo', 'population': 4550203.0, 'lat': 36.146896707500076, 'lon': 37.395163172285166}, {'name': 'Idleb', 'population': 1746525.75, 'lat': 35.857696777500166, 'lon': 36.57087950186119}, {'name': 'Lattakia', 'population': 1519531.0, 'lat': 35.580780029000096, 'lon': 35.99093034807403}]


In [66]:
# ETL Transform
# 人口値をCircleMarkerの半径へ変換

for item in population_results:
    item["radius"] = item["population"] / 300000

print(population_results)

[{'name': 'Aleppo', 'population': 4550203.0, 'lat': 36.146896707500076, 'lon': 37.395163172285166, 'radius': 15.167343333333333}, {'name': 'Idleb', 'population': 1746525.75, 'lat': 35.857696777500166, 'lon': 36.57087950186119, 'radius': 5.8217525}, {'name': 'Lattakia', 'population': 1519531.0, 'lat': 35.580780029000096, 'lon': 35.99093034807403, 'radius': 5.065103333333333}]


In [67]:
# Dark Matter Basemap

tiles_dark_nolabels = 'https://{s}.basemaps.cartocdn.com/dark_nolabels/{z}/{x}/{y}{r}.png'
attr_dark = '&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/attributions">CARTO</a>'

m = folium.Map(
    location=[35.8, 37.0],
    zoom_start=7,
    tiles=tiles_dark_nolabels,
    attr=attr_dark
)

In [68]:
# ETL Load
# 国境線　太さ1
folium.GeoJson(
    'syr_admin0.geojson',
    name='Country',
    style_function=lambda x: {'color': 'white', 'weight': 1, 'fillOpacity': 0}
).add_to(m)

# 地域線　太さ1
folium.GeoJson(
    'syr_admin1.geojson',
    name='Governorate',
    style_function=lambda x: {'color': 'white', 'weight': 1, 'fillOpacity': 0}
).add_to(m)

In [69]:
# ETL Transform
# Folium表示用に必要な列だけ残す

target_admins_map = target_admins[["adm1_name", "geometry"]]

In [70]:
# ETL Load
# 対象Governorateを薄く塗る

folium.GeoJson(
    target_admins_map,
    style_function=lambda x: {
        "fillColor": "#b7d86f",
        "fillOpacity": 0.15,
        "color": "white",
        "weight": 1
    }
).add_to(m)

In [71]:
# シリア14地域の名入れ
# この地図専用で、微妙に位置変え

governorates = {
    "Aleppo": [36.3, 37.5],
    "Al-Hasakeh": [36.5, 40.7],
    "Ar-Raqqa": [36.0, 39.0],
    "As-Sweida": [32.8, 36.9],
    "Daraa": [32.9, 36.2],
    "Deir-ez-Zor": [35.1, 40.5],
    "Damascus": [33.7, 36.7],
    "Hama": [35.2, 37.0],
    "Homs": [34.5, 38.3],
    "Idleb": [35.7, 36.7],
    "Lattakia": [35.7, 36.0],
    "Quneitra": [33.1, 35.9],
    "Rural Damascus": [33.5, 37.5],
    "Tartous": [34.9, 36.1]
}

for name, coords in governorates.items():
    folium.Marker(
        location=coords,
        icon=folium.DivIcon(
            html=f'''<div style="
                font-size: 12pt; 
                color: white; 
                font-weight: bold;
                white-space: nowrap;
                text-align: center;
                width: 100px;
                margin-left: -50px;
                ">{name}</div>'''
        )
    ).add_to(m)

In [72]:
# ETL Load
# 人口合計をCircleMarkerとラベルで地図へ可視化

for item in population_results:
    folium.CircleMarker(
        location=[item["lat"], item["lon"]],
        radius=item["radius"],
        color="#b7d86f",
        fill=True,
        fillColor="#b7d86f",
        fillOpacity=0.65,
        weight=2,
        popup=f"{item['name']}: {item['population']:,.0f} people"
    ).add_to(m)

In [73]:
# ETL Load
# タイトル・説明ボックスをHTMLとして地図に追加

title_html = '''
             <div style="position: fixed; 
                         top: 20px; left: 50px; width: 390px; height: 165px; 
                         background-color: rgba(0,0,0,0.75); color: white;
                         z-index:9000; font-size:14px;
                         border:1px solid white; border-radius:8px; padding: 12px;
                         box-shadow: 0 0 15px rgba(0,0,0,0.5);">
             <b style="font-size: 16px;">Northwest Syria</b><br>
             <span style="color: #b7d86a;">Estimated Population by Governorate (WorldPop 2026)</span><br>
             <small style="display: block; margin-top: 6px; line-height: 1.3; color: #eee;">
                Population totals were calculated from WorldPop 2026 raster data using zonal statistics.
                Circle size represents estimated population by governorate.
             </small>
             <div style="margin-top: 10px; font-size: 11px; color: #bbb; border-top: 1px solid #444; padding-top: 6px;">
             Source: <a href="https://hub.worldpop.org/geodata/summary?id=75632" target="_blank" style="color: #3498db; text-decoration: none;">WorldPop 2026</a><br>
             Method: Zonal Statistics / Proportional Symbol Map
             </div>
             </div>
             '''

m.get_root().html.add_child(folium.Element(title_html))

In [74]:
# ETL Load
# 人口値ラベルを地図へ追加

for item in population_results:
    folium.Marker(
        location=[item["lat"], item["lon"]],
        icon=folium.DivIcon(
            html=f'''
            <div style="
                color:white;
                font-size:12pt;
                font-weight:bold;
                white-space: nowrap;
                text-shadow: 0 0 4px black;
            ">
            {item["population"]/1000000:.2f} M
            </div>
            '''
        )
    ).add_to(m)

In [75]:
# ETL Load
# 凡例ボックスを地図へ追加

legend_html = '''
<div style="
position: fixed;
bottom: 40px;
right: 40px;
width: 180px;
background-color: rgba(0,0,0,0.75);
color: white;
border: 1px solid white;
border-radius: 8px;
padding: 10px;
font-size: 12px;
z-index: 9999;
">

<b>Legend</b><br><br>

● Circle size<br>
= Population<br><br>

Source:<br>
WorldPop 2026

</div>
'''

m.get_root().html.add_child(
    folium.Element(legend_html)
)

In [76]:
# ETL Load
# HTMLマップとして保存

m.save("07_syria_NW3_zonalstatistics.html")